In [1]:
%pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 1.8 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: ml_dtypes━━━━━━━━━━━━━━━ 0/2 [protobuf]
    Found existing installation: ml-dtypes 0.3.2 0/2 [protobuf]
    Uninstalling ml-dtypes-0.3.2:━━━━━━━━━━━ 0/2 [protobuf]
      Successfully uninstalled ml-dtypes-0.3.20m 0/2 [protobuf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ml_dtypes]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import onnx
import onnxruntime as ort
import time

device = "cpu"  # we'll deliberately use CPU to demonstrate quantisation gains
torch.manual_seed(42)

In [2]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
model.load_state_dict(torch.load("flowers102_resnet18.pth", map_location=device))
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

# Task 1 — Export to ONNX and Verify

**Part A — Export.**

1. Define an example input matching the validation pipeline:

```python
example = torch.randn(1, 3, 224, 224)
```

2. Export to ONNX with **explicit dynamic batch axis**:

```python
torch.onnx.export(
    model, example, "flowers_resnet18.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
```

3. Validate the export:

```python
onnx_model = onnx.load("flowers_resnet18.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")
```

4. Print the file size of the exported model in MB.

In [3]:
example = torch.randn(1, 3, 224, 224)

In [4]:
torch.onnx.export(
    model, example, "flowers_resnet18.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)

In [5]:
onnx_model = onnx.load("flowers_resnet18.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")

ONNX model is valid.


In [6]:
import os

file_path = "flowers_resnet18.onnx"
size_bytes = os.path.getsize(file_path)
size_mb = size_bytes / (1024 * 1024)

print(f"Model file size: {size_mb:.2f} MB")


Model file size: 42.83 MB


**Part B — Numerical equivalence check.**

Confirm the exported ONNX model produces the same outputs as the original PyTorch model.

1. Load the ONNX model into an ONNX Runtime session:

```python
session = ort.InferenceSession("flowers_resnet18.onnx")
```

2. Take **8 random validation images**, run them through both models, and compute the maximum absolute difference between their outputs.
3. Assert the difference is below `1e-4`. If not, investigate (different normalisation, dropout still on, etc.).

In [8]:
session = ort.InferenceSession("flowers_resnet18.onnx")

In [9]:
# Take 8 random validation images (here using dummy tensors for illustration)
# In practice, you should sample from your validation DataLoader
images = torch.randn(8, 3, 224, 224)

# Get outputs from the PyTorch model
with torch.no_grad():
    torch_outputs = model(images).cpu().numpy()

# Prepare inputs for ONNX Runtime (convert to numpy float32)
onnx_inputs = {"input": images.cpu().numpy().astype(np.float32)}

# Get outputs from the ONNX model
onnx_outputs = session.run(["logits"], onnx_inputs)[0]

# Compute maximum absolute difference between PyTorch and ONNX outputs
max_diff = np.max(np.abs(torch_outputs - onnx_outputs))
print("Max absolute difference:", max_diff)

# Assert that the difference is below the threshold
assert max_diff < 1e-4, "Outputs differ too much! Check normalization or eval mode."


Max absolute difference: 4.2915344e-06


# Task 2 — Build an Inference Pipeline

Create a clean inference-only function in a separate Python file `inference.py` that:

- Loads the ONNX model once
- Accepts a path to an image file
- Applies the same preprocessing used at training (resize → centre crop → normalise with ImageNet stats)
- Returns the top-3 predicted classes with probabilities

Suggested skeleton:

```python
import numpy as np, onnxruntime as ort
from PIL import Image

class FlowerClassifier:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(onnx_path)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    def preprocess(self, image_path):
        img = Image.open(image_path).convert("RGB").resize((232, 232))
        # centre crop to 224x224
        left = (232 - 224) // 2
        img = img.crop((left, left, left + 224, left + 224))
        x = np.asarray(img, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
        return ((x - self.mean) / self.std).astype(np.float32)

    def predict(self, image_path, k=3):
        x = self.preprocess(image_path)
        logits = self.session.run(None, {"input": x})[0][0]
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        top = np.argsort(probs)[::-1][:k]
        return [(int(i), float(probs[i])) for i in top]
```

In your notebook:
1. Import this class and run inference on 5 test images. Print the top-3 predictions for each.
2. Verify the predictions match what your PyTorch model produces for the same images.


In [11]:
from inference import FlowerClassifier

clf = FlowerClassifier("flowers_resnet18.onnx")

test_images = ["jpg/image_00001.jpg", "jpg/image_00002.jpg", "jpg/image_00003.jpg", "jpg/image_00004.jpg", "jpg/image_00005.jpg"]

for path in test_images:
    preds = clf.predict(path, k=3)
    print(path, preds)


jpg/image_00001.jpg [(76, 0.9883971810340881), (70, 0.00954331737011671), (9, 0.000552130164578557)]
jpg/image_00002.jpg [(76, 0.9605329632759094), (12, 0.005164716858416796), (53, 0.004820137284696102)]
jpg/image_00003.jpg [(76, 0.9019895195960999), (70, 0.026502693071961403), (81, 0.010114019736647606)]
jpg/image_00004.jpg [(37, 0.36468613147735596), (76, 0.19951769709587097), (39, 0.04038744047284126)]
jpg/image_00005.jpg [(76, 0.9922913908958435), (12, 0.0013856962323188782), (61, 0.0006590725970454514)]


In [15]:
# 1. Define the model architecture
model = models.resnet18(num_classes=102)  # Flowers102 dataset has 102 classes

# 2. Load the trained weights
state_dict = torch.load("flowers102_resnet18.pth", map_location="cpu")
model.load_state_dict(state_dict)

# 3. Set to evaluation mode
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [16]:
# Load ONNX model
session = ort.InferenceSession("flowers_resnet18.onnx")

# Preprocess function (same as training)
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

def preprocess(image_path):
    img = Image.open(image_path).convert("RGB").resize((232, 232))
    left = (232 - 224) // 2
    img = img.crop((left, left, left + 224, left + 224))
    x = np.asarray(img, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    return ((x - mean) / std).astype(np.float32)

# Compare predictions
test_images = [
    "jpg/image_00001.jpg",
    "jpg/image_00002.jpg",
    "jpg/image_00003.jpg",
    "jpg/image_00004.jpg",
    "jpg/image_00005.jpg"
]

for path in test_images:
    x = preprocess(path)

    # PyTorch output
    with torch.no_grad():
        torch_logits = model(torch.from_numpy(x)).cpu().numpy()[0]

    # ONNX output
    onnx_logits = session.run(None, {"input": x})[0][0]

    # Difference
    diff = np.max(np.abs(torch_logits - onnx_logits))
    print(f"{path} → Max abs diff: {diff:.6f}")
    assert diff < 1e-4, "Mismatch detected!"


jpg/image_00001.jpg → Max abs diff: 0.000007
jpg/image_00002.jpg → Max abs diff: 0.000007
jpg/image_00003.jpg → Max abs diff: 0.000006
jpg/image_00004.jpg → Max abs diff: 0.000009
jpg/image_00005.jpg → Max abs diff: 0.000010


# Task 3 — Quantise to INT8 and Benchmark All Three Variants

Apply post-training dynamic quantisation, check its quality, and benchmark the three variants (PyTorch, FP32 ONNX, INT8 ONNX) on the same hardware.

1. Apply dynamic quantisation and report the resulting file size and size ratio vs FP32 ONNX:

```python
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="flowers_resnet18.onnx",
    model_output="flowers_resnet18.int8.onnx",
    weight_type=QuantType.QInt8,
)
```

2. Load the quantised model into a new ONNX Runtime session, run it on the validation set, and compare against the FP32 ONNX model: report the **max** and **mean absolute difference** between outputs, and the **test-set accuracy** of both. In a short markdown cell, comment on the accuracy-vs-size trade-off.
3. Benchmark latency for all three variants on a single image, averaging over 100 runs with `time.perf_counter()`, and fill in the table. Then add a 2–3 sentence comment on whether the speedup matched your expectations and where most of the gain came from.

| Model | File size (MB) | Avg latency (ms) | Speedup vs PyTorch |
|---|---|---|---|
| PyTorch (FP32) | … | … | 1.00× |
| ONNX (FP32) | … | … | … |
| ONNX (INT8) | … | … | … |


In [18]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="flowers_resnet18.onnx",
    model_output="flowers_resnet18.int8.onnx",
    weight_type=QuantType.QInt8,
)


In [19]:
fp32_size = os.path.getsize("flowers_resnet18.onnx") / (1024*1024)
int8_size = os.path.getsize("flowers_resnet18.int8.onnx") / (1024*1024)

print(f"FP32 ONNX size: {fp32_size:.2f} MB")
print(f"INT8 ONNX size: {int8_size:.2f} MB")
print(f"Size ratio (INT8/FP32): {int8_size/fp32_size:.2f}")


FP32 ONNX size: 42.83 MB
INT8 ONNX size: 10.76 MB
Size ratio (INT8/FP32): 0.25


In [24]:
from torchvision import transforms

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

val_tf = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])


In [25]:
from torchvision import datasets

val_ds = datasets.Flowers102(
    root="data",
    split="val",
    download=True,
    transform=val_tf
)

test_ds = datasets.Flowers102(
    root="data",
    split="test",
    download=True,
    transform=val_tf
)


100%|████████████████████████| 344862509/344862509 [01:02<00:00, 5534404.52it/s]


Extracting data/flowers-102/102flowers.tgz to data/flowers-102


100%|█████████████████████████████████████| 502/502 [00:00<00:00, 213977.70it/s]


100%|████████████████████████████████| 14989/14989 [00:00<00:00, 7426866.23it/s]


In [26]:
from torch.utils.data import DataLoader

batch_size = 32

val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

print("Validation samples:", len(val_ds))
print("Test samples:", len(test_ds))


Validation samples: 1020
Test samples: 6149


In [28]:
# Run FP32 ONNX on validation set
fp32_outputs = []
y_true = []

for x, y in val_loader:
    x_np = x.numpy()
    fp32_out = session_fp32.run(None, {"input": x_np})[0]
    fp32_outputs.append(fp32_out)
    y_true.append(y.numpy())

fp32_outputs = np.concatenate(fp32_outputs)
y_true = np.concatenate(y_true)

# Accuracy
fp32_preds = np.argmax(fp32_outputs, axis=1)
fp32_acc = (fp32_preds == y_true).mean()
print("FP32 accuracy:", fp32_acc)

# Latency benchmark (single image, 100 runs)
x_single, _ = next(iter(val_loader))
x_single = x_single[0].unsqueeze(0).numpy()

# PyTorch latency
with torch.no_grad():
    start = time.perf_counter()
    for _ in range(100):
        _ = model(torch.from_numpy(x_single))
    end = time.perf_counter()
    pytorch_time = (end - start) / 100

# ONNX FP32 latency
start = time.perf_counter()
for _ in range(100):
    _ = session_fp32.run(None, {"input": x_single})
end = time.perf_counter()
onnx_fp32_time = (end - start) / 100

print("PyTorch avg latency:", pytorch_time)
print("ONNX FP32 avg latency:", onnx_fp32_time)


FP32 accuracy: 0.865686274509804
PyTorch avg latency: 0.0675734366197139
ONNX FP32 avg latency: 0.02400910580996424


| Model          | File size (MB) | Avg latency (ms) | Speedup vs PyTorch |
|----------------|----------------|------------------|--------------------|
| PyTorch (FP32) | 42.91          | 67               | 1.00×              |
| ONNX (FP32)    | 42.83          | 24               | 2.8×               |
| ONNX (INT8)    | 10.76          | Unsupported      | N/A                |

**Comment:**  
Quantisation reduces model size significantly (≈4× smaller) and usually preserves accuracy within <1% difference. On macOS CPU backend, INT8 inference is not supported, so only file size reduction was measured.

**Latency Benchmark:**  
ONNX FP32 was about 2.8× faster than PyTorch FP32, which matched expectations. The speedup mainly came from ONNX Runtime’s optimised operator implementations compared to PyTorch’s eager execution.


In [32]:
!ls


README.md
__pycache__
data
flowers102_resnet18.pth
flowers_resnet18.int8.onnx
flowers_resnet18.onnx
inference.py
jpg
m6-08-model-conversions-inferencing.ipynb
